# Geomagnetism and particle motion: report preparation

This notebook contains examples of the software tools needed for Report 1, plus an optional part at the end to let you make your own geomagnetic field model.

The examples show how to:

- download Swarm magnetic field data
- evaluate IGRF and read its dipole coefficients
- plot data on global and polar maps
- prepare dataset of Space Shuttle computer anomalies
- integrate the equation of motion of a charged particle


## Setup

Magnetic field values from Swarm and the `ppigrf` module are in nT. Altitudes passed to `ppigrf` are in km. The particle calculation uses SI units: tesla, metres, seconds, coulombs, and kilograms.

For interactive Matplotlib figures in JupyterLab, uncomment `%matplotlib widget` below.

In [ ]:
# %matplotlib widget

from io import StringIO
from pathlib import Path
import warnings

import cartopy.crs as ccrs
import matplotlib.path as mpath
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ppigrf
import course_tools.geomagnetism

from ppigrf.ppigrf import read_shc
from scipy.integrate import solve_ivp
from viresclient import SwarmRequest

# suppress annoying warning:
warnings.filterwarnings("ignore", message  = "invalid value encountered in create_collection", category = RuntimeWarning, module   = r"shapely\.creation")

plt.rcParams.update({"figure.figsize": (10, 5), "axes.grid": True})

# 1. Swarm measurements and IGRF

## Download Swarm NEC data

The example below requests one day of Swarm A magnetic-field measurements at 30-second cadence. `B_NEC` contains components in the north, east, centre directions. Centre is downward toward Earth.

One day is enough for the examples later in this notebook. Choose a longer interval for the report. A local installation may ask you to authenticate with VirES the first time (see https://viresclient.readthedocs.io/en/latest/access_token.html). 

In [ ]:
SWARM_START = "2024-05-08T00:00:00"
SWARM_END   = "2024-05-09T00:00:00"

swarm_request = SwarmRequest()
swarm_request.set_collection("SW_OPER_MAGA_LR_1B")
swarm_request.set_products(measurements=["B_NEC"], sampling_step="PT30S")

swarm_result = swarm_request.get_between(SWARM_START, SWARM_END, asynchronous=False, show_progress=True)

swarm_measurements = swarm_result.as_dataframe(expand=True).sort_index()
swarm_measurements.index.name = "time_utc"
swarm_measurements.head()

The returned table includes time, geographic latitude and longitude, geocentric radius, and the three NEC field components. 

## Evaluate IGRF with `ppigrf`

This example evaluates the full IGRF at one location and time. Longitude is positive east, latitude is geodetic, and altitude is measured above the ellipsoid. The output order is radial, southward, eastward.

In [ ]:
EXAMPLE_TIME       = pd.Timestamp("2025-01-01").to_pydatetime()
EXAMPLE_LONGITUDE  = 12.57     # degrees east
EXAMPLE_COLATITUDE = 55.68     # colatitude in degrees
EXAMPLE_RADIUS_KM  = 6831.2    # radius in km

B_r, B_theta, B_phi = ppigrf.igrf_gc(EXAMPLE_RADIUS_KM, EXAMPLE_COLATITUDE, EXAMPLE_LONGITUDE, EXAMPLE_TIME)

print(f"B east   = {B_phi.item():.2f} nT")
print(f"B north  = {-B_theta.item():.2f} nT")
print(f"B radial = {B_r.item():.2f} nT")

When comparing with Swarm NEC data, pay attention to both component order and sign: Swarm C is downward, whereas `ppigrf` returns the radial (outward) component.

When using `ppigrf` it is best to stick to a single time, instead of passing the exact Swarm measurement times. The main magnetic field changes very slowly, so the time only needs to be updated if you look at measurements that are years apart

## Extract the IGRF dipole coefficients

`read_shc()` returns the cosine coefficients $g_n^m$ and sine coefficients $h_n^m$ as Pandas DataFrames. The columns are indexed by degree $n$ and order $m$. The three degree-one coefficients define the centered dipole.

In [ ]:
g_coefficients, h_coefficients = read_shc()

DIPOLE_EPOCH = pd.Timestamp("2025-01-01")

g10 = g_coefficients.loc[DIPOLE_EPOCH, (1, 0)]
g11 = g_coefficients.loc[DIPOLE_EPOCH, (1, 1)]
h11 = h_coefficients.loc[DIPOLE_EPOCH, (1, 1)]

print(f"g10 = {g10:.1f} nT")
print(f"g11 = {g11:.1f} nT")
print(f"h11 = {h11:.1f} nT")

The coefficient table contains five-year epochs. Change `DIPOLE_EPOCH` to inspect another tabulated epoch. Use the equations from class to turn these coefficients into a dipole field or pole position.

# 2. Cartopy map examples

The values and points below are artificial and only demonstrate the plotting syntax. Replace them with the quantities needed in the report.

In [ ]:
map_longitude = np.linspace(-180, 180, 73)
map_latitude  = np.linspace(-90, 90, 37)
longitude_grid, latitude_grid = np.meshgrid(map_longitude, map_latitude)

example_values = np.cos(np.radians(latitude_grid)) * np.cos(2 * np.radians(longitude_grid))

example_point_longitude = np.array([-70.0, 20.0, 140.0])
example_point_latitude  = np.array([-25.0, 10.0, 55.0])

## Global map

In [ ]:
figure = plt.figure(figsize=(11, 5))
axis   = figure.add_subplot(1, 1, 1, projection=ccrs.Robinson())

axis.set_global()
axis.coastlines(resolution="110m", linewidth=0.7)

axis.gridlines(linewidth=0.4, color="0.5")

contours = axis.contourf(longitude_grid, latitude_grid, example_values, levels=15, transform=ccrs.PlateCarree(), cmap="coolwarm")

axis.scatter(example_point_longitude, example_point_latitude, transform=ccrs.PlateCarree(), color="black", s=30, label="Example points")

axis.set_title("Global map example")
axis.legend()
figure.colorbar(contours, ax=axis, shrink=0.75, label="Example values")
figure.tight_layout()

## North polar map

In [ ]:
figure = plt.figure(figsize=(7, 7))
axis   = figure.add_subplot(1, 1, 1, projection=ccrs.NorthPolarStereo())

axis.set_extent([-180, 180, 50, 90], crs=ccrs.PlateCarree())
axis.coastlines(resolution="110m", linewidth=0.7)
axis.gridlines(linewidth=0.4, color="0.5")

boundary_angle = np.linspace(0, 2 * np.pi, 100)
map_boundary   = np.column_stack([0.5 + 0.5 * np.sin(boundary_angle),
                                  0.5 + 0.5 * np.cos(boundary_angle)])
axis.set_boundary(mpath.Path(map_boundary), transform=axis.transAxes)

contours = axis.contourf(longitude_grid, latitude_grid, example_values, levels=15, transform=ccrs.PlateCarree(), cmap="coolwarm")

axis.scatter(np.array([-100.0, 30.0, 150.0]), np.array([70.0, 80.0, 60.0]), transform=ccrs.PlateCarree(), color="black", s=30, label="Example points")

axis.set_title("North polar map example")
axis.legend()
figure.colorbar(contours, ax=axis, shrink=0.75, label="Example values")
figure.tight_layout()

`transform=ccrs.PlateCarree()` tells Cartopy that the data coordinates are geographic longitude and latitude, even though the displayed map uses another projection.

# 3. Space Shuttle anomaly data

The file `../data/anom5j.xls` is a local copy of the [NOAA spacecraft-anomaly database](https://www.ngdc.noaa.gov/stp/space-weather/satellite-data/spacecraft-anomalies/data/anom5j.xls). 

The code below reads the file, selects identifiers beginning with `STS-`, constructs a timestamp and signed coordinates, and retains a small set of readable columns.

In [ ]:
ANOMALY_FILE = Path("../data/anom5j.xls")

if not ANOMALY_FILE.exists():
    ANOMALY_FILE = Path("data/anom5j.xls")

excel_reader_messages = StringIO()

anomaly_records = pd.read_excel( ANOMALY_FILE, sheet_name="anom5j", engine="xlrd", engine_kwargs={"logfile": excel_reader_messages})

shuttle_records = anomaly_records[anomaly_records["BIRD"].str.startswith("STS-", na=False)].copy()

utc_hhmm = shuttle_records["STIMEU"].astype(int).astype(str).str.zfill(4)

time_utc = (pd.to_datetime(shuttle_records["ADATE"]) + pd.to_timedelta(utc_hhmm.str[:2].astype(int), unit="h") + pd.to_timedelta(utc_hhmm.str[2:].astype(int), unit="m"))

latitude_deg = np.where(shuttle_records["NS"].eq("S"),-shuttle_records["LAT"],shuttle_records["LAT"])

longitude_deg = np.where(shuttle_records["EW"].eq("W"),-shuttle_records["LON"],shuttle_records["LON"])

shuttle_anomalies = pd.DataFrame({
    "mission":       shuttle_records["BIRD"],
    "time_utc":      time_utc,
    "latitude_deg":  latitude_deg,
    "longitude_deg": longitude_deg,
    "altitude_km":   shuttle_records["ALT"],
}).sort_values("time_utc").reset_index(drop=True)

shuttle_anomalies

Use the global Cartopy pattern above to decide how to present the event locations and the magnetic-field quantity used for comparison.

# 4. Particle integration

With no electric field, the non-relativistic equation of motion is

$$
\frac{d\mathbf r}{dt}=\mathbf v,
\qquad
\frac{d\mathbf v}{dt}=\frac{q}{m}\,\mathbf v\times\mathbf B(\mathbf r).
$$

The cells below are a runnable zero-field syntax check. For the report, replace the magnetic field and initial conditions with your physical model. Keep all quantities in SI units.

## Magnetic field and initial conditions: student input

Write the three Cartesian components of your field in `magnetic_field`. The position passed to it is the current $(x,y,z)$ position in metres, and the return value must be $(B_x,B_y,B_z)$ in tesla.

In [ ]:
ELEMENTARY_CHARGE = 1.602176634e-19       # C
PROTON_MASS       = 1.67262192369e-27     # kg
EARTH_RADIUS      = 6371.2e3              # m


def magnetic_field(position):
    x, y, z = position

    # Replace this zero field with your magnetic-field equations.
    Bx = 0.0
    By = 0.0
    Bz = 0.0

    return np.array([Bx, By, Bz])


particle_charge = ELEMENTARY_CHARGE
particle_mass   = PROTON_MASS

# Replace these placeholder initial conditions and time scales.
position_0 = np.array([0.0, 0.0, 0.0])       # m
velocity_0 = np.array([1.0, 0.0, 0.0])       # m/s
duration   = 1.0                              # s
time_step  = 0.01                             # s

Choose `time_step` from the shortest physical time scale in the calculation. In a magnetic field this is normally the shortest gyroperiod encountered by the particle.

## Forward Euler method

The Euler loop displays the update explicitly. Both derivatives are evaluated at the beginning of each step.

In [ ]:
number_of_steps = int(np.floor(duration / time_step))
time_euler      = np.arange(number_of_steps + 1) * time_step
position_euler  = np.zeros((number_of_steps + 1, 3))
velocity_euler  = np.zeros((number_of_steps + 1, 3))

position_euler[0] = position_0
velocity_euler[0] = velocity_0

for index in range(number_of_steps):
    position = position_euler[index]
    velocity = velocity_euler[index]
    B        = magnetic_field(position)

    dposition_dt = velocity
    dvelocity_dt = particle_charge / particle_mass * np.cross(velocity, B)

    position_euler[index + 1] = position + dposition_dt * time_step
    velocity_euler[index + 1] = velocity + dvelocity_dt * time_step

## SciPy DOP853 method

`solve_ivp` expects a function whose inputs are time and the complete state. The first three state values are position and the last three are velocity. The two derivative lines are the two equations written above.

In [ ]:
def equation_of_motion(time, state):
    position = state[:3]
    velocity = state[3:]
    B        = magnetic_field(position)

    dposition_dt = velocity
    dvelocity_dt = particle_charge / particle_mass * np.cross(velocity, B)

    return np.concatenate([dposition_dt, dvelocity_dt])

In [ ]:
initial_state = np.concatenate([position_0, velocity_0])
output_times  = time_euler

solution = solve_ivp(
    equation_of_motion,
    (output_times[0], output_times[-1]),
    initial_state,
    method="DOP853",
    t_eval=output_times,
    max_step=time_step,
    rtol=1e-9,
    atol=1e-12,
)

if not solution.success:
    raise RuntimeError(solution.message)

time_dop853     = solution.t
position_dop853 = solution.y[:3].T
velocity_dop853 = solution.y[3:].T

The resulting arrays have matching rows: one time, position, and velocity per output sample. 

For the report, you must supply the magnetic-field equations, construct physical initial conditions, choose a justified step size and duration, and decide how to measure gyro- and guiding-centre motion. 

# Optional: fit an internal field model to Swarm data

This section builds a small degree-one model from a small subset of the downloaded Swarm magnetic field measurements. 

The spherical-harmonic basis calculation is supplied by `course_tools.geomagnetism.design_matrix`. The least-squares problem is

$$
Gm=d,
$$

where each row of $G$ describes how a Gauss coefficient contributes to one measured field component, $d$ contains the Swarm measurements, and $m$ contains the coefficients to be estimated.

In [ ]:
MODEL_DEGREE        = 1
NUMBER_OF_MODEL_DATA = 60
RANDOM_SEED         = 1

required_columns = [
    "Radius",
    "Latitude",
    "Longitude",
    "B_NEC_N",
    "B_NEC_E",
    "B_NEC_C",
]

available_model_data = swarm_measurements.dropna(subset=required_columns)

# get an approximately uniformly distributed subset of the data by randomly sampling weighted by cos(latitudue)
sampling_weights     = np.cos(np.deg2rad(available_model_data["Latitude"]))
sampling_weights     = sampling_weights / sampling_weights.sum()
random_generator = np.random.default_rng(RANDOM_SEED)
selected_rows    = random_generator.choice(
    len(available_model_data),
    size=NUMBER_OF_MODEL_DATA,
    replace=False,
    p=sampling_weights,
)

model_data = available_model_data.iloc[selected_rows].copy()

model_data["measured_magnitude_nT"] = np.sqrt(model_data["B_NEC_N"]**2 + model_data["B_NEC_E"]**2 + model_data["B_NEC_C"]**2)

print(f"Available measurements: {len(available_model_data)}")
print(f"Selected measurements:  {len(model_data)}")
model_data[required_columns + ["measured_magnitude_nT"]].head()

## Display the selected measurements

The colors show measured field magnitude. The points also reveal the geographical sampling of the fit.

In [ ]:
figure = plt.figure(figsize=(11, 5))
axis   = figure.add_subplot(1, 1, 1, projection=ccrs.Robinson())

axis.set_global()
axis.coastlines(resolution="110m", linewidth=0.7)
axis.gridlines(linewidth=0.4, color="0.5")

image = axis.scatter(model_data["Longitude"], model_data["Latitude"], c=model_data["measured_magnitude_nT"], transform=ccrs.PlateCarree(), cmap="viridis", s=30)

axis.set_title("Swarm measurements selected for the fit")
figure.colorbar(image, ax=axis, shrink=0.75, label="$|B|$ [nT]")
figure.tight_layout()

## Construct the design matrix $G$

For degree one there are three unknown coefficients. The matrix has three blocks of rows: all north-component equations, followed by east and centre.

In [ ]:
G, cosine_keys, sine_keys = course_tools.geomagnetism.design_matrix(
    model_data["Radius"].to_numpy(),
    model_data["Latitude"].to_numpy(),
    model_data["Longitude"].to_numpy(),
    MODEL_DEGREE,
)

coefficient_names = ([f"g({degree},{order})" for degree, order in cosine_keys] + [f"h({degree},{order})" for degree, order in sine_keys])

print("G shape:", G.shape)
pd.DataFrame(G[:5], columns=coefficient_names)

## Construct the data vector $d$

The values are stacked in the same north, east, centre order as the rows of $G$.

In [ ]:
d = np.concatenate([
    model_data["B_NEC_N"].to_numpy(),
    model_data["B_NEC_E"].to_numpy(),
    model_data["B_NEC_C"].to_numpy(),
])

component_labels = np.repeat(["north", "east", "centre"], len(model_data))

d_table = pd.DataFrame({"component": component_labels, "measured_nT": d})

print("d shape:", d.shape)
d_table.groupby("component", sort=False).head(3)

## Solve $Gm=d$

`np.linalg.lstsq` finds the coefficient vector that minimizes the sum of squared component residuals.

In [ ]:
m = np.linalg.lstsq(G, d, rcond=None)[0]

fitted_coefficients = pd.Series(m, index = coefficient_names, name="fitted coefficient [nT]")

fitted_coefficients

## Evaluate the fitted model at the measurement locations

Matrix multiplication gives the fitted north, east and centre components in exactly the same order as $d$.

In [ ]:
fitted_values = G @ m
fitted_north, fitted_east, fitted_centre = np.split(fitted_values, 3)

model_data["model_B_NEC_N"] = fitted_north
model_data["model_B_NEC_E"] = fitted_east
model_data["model_B_NEC_C"] = fitted_centre

model_data["model_magnitude_nT"] = np.sqrt(fitted_north**2 + fitted_east**2 + fitted_centre**2)

model_data["magnitude_residual_nT"] = ( model_data["measured_magnitude_nT"] - model_data["model_magnitude_nT"])

model_data[["measured_magnitude_nT", "model_magnitude_nT", "magnitude_residual_nT"]].head()

## Evaluate the fitted model on a global grid

The map uses the mean measurement radius. A coarse ten-degree grid is sufficient for this degree-one example.

In [ ]:
model_time = (pd.Timestamp(SWARM_START) + (pd.Timestamp(SWARM_END) - pd.Timestamp(SWARM_START)) / 2).to_pydatetime()

map_radius_m = model_data["Radius"].mean()

model_longitude = np.arange(-180, 181, 10)
model_latitude  = np.arange(-85, 86, 10)
model_longitude_grid, model_latitude_grid = np.meshgrid(model_longitude, model_latitude)

flat_longitude = model_longitude_grid.ravel()
flat_latitude  = model_latitude_grid.ravel()
flat_radius_m  = np.full(flat_longitude.size, map_radius_m)

G_map, _, _ = course_tools.geomagnetism.design_matrix(flat_radius_m, flat_latitude, flat_longitude, MODEL_DEGREE)

model_map_north, model_map_east, model_map_centre = np.split(G_map @ m, 3)

model_map_magnitude = np.sqrt(model_map_north**2 + model_map_east**2 + model_map_centre**2).reshape(model_latitude_grid.shape)

print("Map grid shape:", model_map_magnitude.shape)

In [ ]:
figure = plt.figure(figsize=(11, 5))
axis   = figure.add_subplot(1, 1, 1, projection=ccrs.Robinson())

axis.set_global()
axis.coastlines(resolution="110m", linewidth=0.7)
axis.gridlines(linewidth=0.4, color="0.5")

image = axis.contourf(
    model_longitude_grid,
    model_latitude_grid,
    model_map_magnitude,
    levels=15,
    transform=ccrs.PlateCarree(),
    cmap="viridis",
)

axis.set_title("Degree-one field fitted to Swarm measurements")
figure.colorbar(image, ax=axis, shrink=0.75, label="$|B|$ [nT]")
figure.tight_layout()

## Compare the fitted model with IGRF

For this geocentric grid, `ppigrf.igrf_gc` returns outward radial, southward and eastward components. The signs below convert them to north, east, centre before calculating magnitude.

In [ ]:
igrf_radial, igrf_south, igrf_east = ppigrf.igrf_gc(
    flat_radius_m / 1000,
    90 - flat_latitude,
    flat_longitude,
    model_time,
)

igrf_north  = -np.asarray(igrf_south).reshape(-1)
igrf_east   =  np.asarray(igrf_east).reshape(-1)
igrf_centre = -np.asarray(igrf_radial).reshape(-1)

igrf_map_magnitude = np.sqrt(igrf_north**2 + igrf_east**2 + igrf_centre**2).reshape(model_latitude_grid.shape)

model_minus_igrf = model_map_magnitude - igrf_map_magnitude

In [ ]:
figure = plt.figure(figsize=(11, 5))
axis   = figure.add_subplot(1, 1, 1, projection=ccrs.Robinson())

axis.set_global()
axis.coastlines(resolution="110m", linewidth=0.7)
axis.gridlines(linewidth=0.4, color="0.5")

difference_limit = np.max(np.abs(model_minus_igrf))
image = axis.contourf(
    model_longitude_grid,
    model_latitude_grid,
    model_minus_igrf,
    levels=np.linspace(-difference_limit, difference_limit, 16),
    transform=ccrs.PlateCarree(),
    cmap="coolwarm",
    extend="both",
)

axis.set_title("Fitted degree-one model minus full IGRF")
figure.colorbar(
    image,
    ax=axis,
    shrink=0.75,
    label="Difference in $|B|$ [nT]",
)
figure.tight_layout()

Further optional explorations:

- plot the fitted values or residuals at the measurement locations
- compare the fitted degree-one coefficients with the IGRF degree-one coefficients
- change the random seed or number of measurements
- increase `MODEL_DEGREE` and identify which spatial structures appear